# Marketing Analytics Notebook: EDA, Funnel Analysis, LTV, CAC, ROI

## Business problem statement
You are supporting a growth/marketing team running multi-channel acquisition campaigns (Organic, Paid Search, Paid Social, Referral, Email).  
Leadership wants a **single, repeatable analysis** to answer:

1. **What does our acquisition + activation funnel look like?** Where do we lose users?
2. **How much do we pay to acquire customers (CAC), overall and by channel?**
3. **What is the estimated customer lifetime value (LTV), overall and by channel/cohort?**
4. **Are campaigns profitable (ROI), overall and by channel?**
5. **What data quality issues exist (missing values, zeros, skew, outliers) that could bias decisions?**

This notebook walks through:
- **EDA** (data validation, missingness, distributions, correlations)
- **Funnel analysis** (step conversion rates and drop-offs)
- **LTV** (simple revenue × tenure proxy + channel & cohort breakdowns)
- **CAC** (cost per converted customer)
- **ROI** (profitability by channel)

> Dataset note: The dataset intentionally includes **missing values and imperfections** to simulate real-world data.


## 1) Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load
df = pd.read_csv("/mnt/data/marketing_funnel_dataset.csv", parse_dates=["signup_date"])

# Quick view
df.head()


## 2) EDA: Structure, Types, and Basic Checks

In [ ]:
df.shape, df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 3) EDA: Missing Values & Data Quality

We'll quantify missing values and visualize which columns are most affected.


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_table

In [ ]:
# Missingness bar chart
missing_table = missing_table[missing_table["missing_count"] > 0].sort_values("missing_count", ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(missing_table.index, missing_table["missing_count"])
plt.title("Missing Values by Column")
plt.xlabel("Count of Missing Values")
plt.ylabel("Column")
plt.tight_layout()
plt.show()


## 4) EDA: Key Distributions (Costs, Revenue, Tenure)

Marketing and revenue data are often **skewed** (many small values, few large ones).  
We’ll inspect distributions and zeros.


In [ ]:
numeric_cols = ["campaign_cost", "impressions", "clicks", "monthly_revenue", "months_active", "lifetime_value"]
df[numeric_cols].describe().T

In [ ]:
# Histograms for key numeric features (ignoring NaNs)
cols_to_plot = ["campaign_cost", "impressions", "clicks", "monthly_revenue", "months_active"]

for col in cols_to_plot:
    plt.figure(figsize=(7, 4))
    x = df[col].dropna()
    plt.hist(x, bins=30)
    plt.title(f"Distribution: {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
# Zero checks
zero_checks = pd.DataFrame({
    "zero_count": (df[["impressions", "clicks"]].fillna(0) == 0).sum(),
    "zero_pct": ((df[["impressions", "clicks"]].fillna(0) == 0).sum() / len(df) * 100).round(2)
})
zero_checks

## 5) EDA: Channel Breakdown

We’ll see volume by channel and note missing/unknown channels.


In [ ]:
channel_counts = df["channel"].fillna("Unknown").value_counts()
channel_counts

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(channel_counts.index.astype(str), channel_counts.values)
plt.title("Users by Acquisition Channel")
plt.xlabel("Channel")
plt.ylabel("Users")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 6) EDA: Correlation Snapshot (Numeric Features)

This is a quick, exploratory view to spot relationships (e.g., clicks vs signups, revenue vs tenure).


In [ ]:
# Prepare numeric frame (coerce non-numeric where needed)
num_df = df.copy()
for c in ["signups", "activated", "converted"]:
    num_df[c] = pd.to_numeric(num_df[c], errors="coerce")

corr_cols = ["impressions", "clicks", "signups", "activated", "converted", "campaign_cost", "monthly_revenue", "months_active", "lifetime_value"]
corr = num_df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title("Correlation Matrix (Numeric)")
plt.tight_layout()
plt.show()

corr

# 7) Funnel Analysis

We’ll define a simple funnel:
1. **Impressions** (has exposure)
2. **Clicks** (engagement)
3. **Signups**
4. **Activated**
5. **Converted** (paying customer)

Because the dataset is user-level, we’ll compute:
- Counts at each step
- Conversion rates between steps
- Channel-specific funnel differences


In [ ]:
# Define funnel step booleans (robust to NaNs)
funnel = pd.DataFrame({
    "Exposed (Impressions>0)": df["impressions"].fillna(0) > 0,
    "Clicked (Clicks>0)": df["clicks"].fillna(0) > 0,
    "Signed Up": df["signups"].fillna(0).astype(int) == 1,
    "Activated": df["activated"].fillna(0).astype(int) == 1,
    "Converted": df["converted"].fillna(0).astype(int) == 1,
})

funnel_counts = funnel.sum().astype(int)
funnel_counts

In [ ]:
# Step-to-step conversion rates
steps = funnel_counts.index.tolist()
counts = funnel_counts.values

rates = []
for i in range(1, len(counts)):
    prev = counts[i-1]
    curr = counts[i]
    rates.append(curr / prev if prev else np.nan)

funnel_rates = pd.DataFrame({
    "from_step": steps[:-1],
    "to_step": steps[1:],
    "from_count": counts[:-1],
    "to_count": counts[1:],
    "conversion_rate": np.round(rates, 4)
})
funnel_rates

In [ ]:
# Funnel bar chart
plt.figure(figsize=(9, 4))
plt.bar(steps, counts)
plt.title("Funnel Counts")
plt.xlabel("Funnel Step")
plt.ylabel("Users")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 7.0) Funnel Visualization (Actual Funnel Shape)

A classic funnel can be visualized by **centering bars** for each step so the widths decrease as users drop off.
This produces a true *funnel-like* shape while staying in pure Matplotlib.


In [ ]:
# "Actual funnel" using centered horizontal bars (no custom colors)
labels = funnel_counts.index.tolist()
values = funnel_counts.values.astype(float)

max_val = np.nanmax(values)
y_pos = np.arange(len(labels))

# Center each bar by shifting left so it is centered around max width
left_offsets = (max_val - values) / 2.0

plt.figure(figsize=(9, 5))
plt.barh(y_pos, values, left=left_offsets)
plt.yticks(y_pos, labels)
plt.gca().invert_yaxis()  # first step at the top
plt.title("Marketing Funnel (Centered Bars)")
plt.xlabel("Users")
plt.tight_layout()
plt.show()


In [ ]:
# Funnel conversion rate chart
plt.figure(figsize=(9, 4))
plt.plot(funnel_rates["to_step"], funnel_rates["conversion_rate"], marker="o")
plt.title("Step-to-step Conversion Rates")
plt.xlabel("To Step")
plt.ylabel("Conversion Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## 7.1) Funnel by Channel

This helps identify which channels bring high-quality users (better activation/conversion), not just volume.


In [ ]:
df2 = df.copy()
df2["channel_clean"] = df2["channel"].fillna("Unknown")

def channel_funnel(group):
    f = pd.DataFrame({
        "Exposed": group["impressions"].fillna(0) > 0,
        "Clicked": group["clicks"].fillna(0) > 0,
        "Signed Up": group["signups"].fillna(0).astype(int) == 1,
        "Activated": group["activated"].fillna(0).astype(int) == 1,
        "Converted": group["converted"].fillna(0).astype(int) == 1,
    })
    return f.sum().astype(int)

channel_funnel_counts = df2.groupby("channel_clean").apply(channel_funnel)
channel_funnel_counts

In [ ]:
# Visualize conversion to "Converted" by channel (Converted / Exposed)
conv_by_channel = (channel_funnel_counts["Converted"] / channel_funnel_counts["Exposed"]).replace([np.inf, -np.inf], np.nan)

plt.figure(figsize=(9, 4))
plt.bar(conv_by_channel.index.astype(str), conv_by_channel.values)
plt.title("Exposed → Converted Rate by Channel")
plt.xlabel("Channel")
plt.ylabel("Conversion Rate")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

conv_by_channel.sort_values(ascending=False)

# 8) LTV (Customer Lifetime Value)

We’ll compute a simple LTV proxy:
\[ \text{LTV} = \text{Monthly Revenue} \times \text{Months Active} \]

Because missing values exist, we’ll create both:
- **Raw LTV** (as-is, may be NaN)
- **Clean LTV** (treat missing revenue/tenure as 0 for conservative estimate)

Then we’ll summarize overall, by channel, and by signup cohort.


In [ ]:
ltv = df.copy()
ltv["monthly_revenue_clean"] = ltv["monthly_revenue"].fillna(0)
ltv["months_active_clean"] = ltv["months_active"].fillna(0)

ltv["ltv_clean"] = ltv["monthly_revenue_clean"] * ltv["months_active_clean"]

ltv[["monthly_revenue", "months_active", "lifetime_value", "ltv_clean"]].head()

In [ ]:
# Overall LTV summary (converted customers only vs all users)
overall = pd.DataFrame({
    "population": ["All users", "Converted users only"],
    "users": [len(ltv), int((ltv["converted"].fillna(0).astype(int)==1).sum())],
    "avg_ltv_clean": [
        ltv["ltv_clean"].mean(),
        ltv.loc[ltv["converted"].fillna(0).astype(int)==1, "ltv_clean"].mean()
    ],
    "median_ltv_clean": [
        ltv["ltv_clean"].median(),
        ltv.loc[ltv["converted"].fillna(0).astype(int)==1, "ltv_clean"].median()
    ]
})
overall

In [ ]:
# LTV by channel
ltv["channel_clean"] = ltv["channel"].fillna("Unknown")
ltv_by_channel = ltv.groupby("channel_clean")["ltv_clean"].agg(["count", "mean", "median", "sum"]).sort_values("mean", ascending=False)
ltv_by_channel

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(ltv_by_channel.index.astype(str), ltv_by_channel["mean"].values)
plt.title("Average LTV (Clean) by Channel")
plt.xlabel("Channel")
plt.ylabel("Average LTV")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
ltv["signup_month"] = ltv["signup_date"].dt.to_period("M").astype(str)
cohort_ltv = ltv.groupby("signup_month")["ltv_clean"].mean().sort_index()

plt.figure(figsize=(10, 4))
plt.plot(cohort_ltv.index, cohort_ltv.values, marker="o")
plt.title("Average LTV (Clean) by Signup Month Cohort")
plt.xlabel("Signup Month")
plt.ylabel("Average LTV")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

cohort_ltv

# 9) CAC (Customer Acquisition Cost)

A common definition:
\[ \text{CAC} = \frac{\text{Total Marketing Cost}}{\text{Number of New Customers}} \]

Here we’ll treat **Converted = 1** as a new paying customer.
We’ll compute CAC:
- Overall
- By channel

We will also show how missing costs impact the calculation (we’ll use 0 for missing cost as a conservative assumption).


In [ ]:
cac = df.copy()
cac["channel_clean"] = cac["channel"].fillna("Unknown")
cac["converted_int"] = cac["converted"].fillna(0).astype(int)

# conservative: missing campaign cost treated as 0
cac["campaign_cost_clean"] = cac["campaign_cost"].fillna(0)

total_cost = cac["campaign_cost_clean"].sum()
total_customers = cac["converted_int"].sum()
overall_cac = total_cost / total_customers if total_customers else np.nan

overall_cac

In [ ]:
# CAC by channel
cost_by_channel = cac.groupby("channel_clean")["campaign_cost_clean"].sum()
cust_by_channel = cac.groupby("channel_clean")["converted_int"].sum()

cac_by_channel = (cost_by_channel / cust_by_channel).replace([np.inf, -np.inf], np.nan).sort_values()
cac_by_channel

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(cac_by_channel.index.astype(str), cac_by_channel.values)
plt.title("CAC by Channel (Cost / Converted Customers)")
plt.xlabel("Channel")
plt.ylabel("CAC")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


# 10) ROI (Return on Investment)

A simple marketing ROI:
\[ ROI = \frac{\text{Revenue} - \text{Cost}}{\text{Cost}} \]

We'll use:
- **Revenue proxy**: clean LTV (conservative)
- **Cost proxy**: campaign_cost_clean

Compute ROI:
- Overall
- By channel


In [ ]:
roi = df.copy()
roi["channel_clean"] = roi["channel"].fillna("Unknown")

roi["campaign_cost_clean"] = roi["campaign_cost"].fillna(0)
roi["monthly_revenue_clean"] = roi["monthly_revenue"].fillna(0)
roi["months_active_clean"] = roi["months_active"].fillna(0)
roi["ltv_clean"] = roi["monthly_revenue_clean"] * roi["months_active_clean"]

total_rev = roi["ltv_clean"].sum()
total_cost = roi["campaign_cost_clean"].sum()
overall_roi = (total_rev - total_cost) / total_cost if total_cost else np.nan

total_rev, total_cost, overall_roi

In [ ]:
# ROI by channel
rev_by_channel = roi.groupby("channel_clean")["ltv_clean"].sum()
cost_by_channel = roi.groupby("channel_clean")["campaign_cost_clean"].sum()

roi_by_channel = ((rev_by_channel - cost_by_channel) / cost_by_channel).replace([np.inf, -np.inf], np.nan).sort_values(ascending=False)
roi_by_channel

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(roi_by_channel.index.astype(str), roi_by_channel.values)
plt.title("ROI by Channel")
plt.xlabel("Channel")
plt.ylabel("ROI")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
